In [1]:
from IPython.terminal.shortcuts.auto_suggest import accept
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict,Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def recipe_search(query:str) -> Dict[str, Any]:

    """ Search the web for the Food Recipe Information"""

    return tavily_client.search(query)

In [3]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

system_prompt = """
You are the Chef who has the recipes to make food with any Ingredients that is available.
Rules:
1. Give the recipe one by one in steps
2. Include timings of the steps(example :- cook on medium heat for 5 minutes, marinate for 30 minutes)
3. Only Use the Ingredients mentioned by the user
4. And if Ingredients are not mentioned then assume User has all the required Ingredients
5. If the User give image of the Ingredient then analyze the image and then list those ingredients
6. If the User give audio asking for recipe give the user the recipe
"""

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    system_prompt=system_prompt,
    tools = [recipe_search],
    checkpointer = InMemorySaver()
)

In [13]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm


duration = 5
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate),
               samplerate=sample_rate, channels=1)

for _ in tqdm(range(duration*10)):
    time.sleep(0.1)
sd.wait()
print("Done.")

buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.75it/s]


Done.


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

In [ ]:
print(uploader.value)

In [ ]:
import base64

uploaded_file = uploader.value[0]

content_mv = uploaded_file["content"]

img_bytes = bytes(content_mv)

img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [4]:
from langchain.messages import HumanMessage

try:
    img_b64
except NameError:
    img_b64 = None

try:
    aud_b64
except NameError:
    aud_b64 = None


while True:
    message_content = []
    user_text = input("Put your query here(or Press Enter for Image reading only)\n")
    if user_text.lower() == "quit":
        print("Chef : Goodbye")
        break

    if user_text:
        message_content.append({"type" : "text", "text" : user_text})

    if img_b64:
        message_content.append(
            {"type" : "image_url",
             "image_url": {"url": f"data:image/png;base64,{img_b64}"}}
        )

    if aud_b64:
        message_content.append(
            {"type": "media", "mime_type": "audio/wav", "data": aud_b64}
        )

    if not message_content:
        print("Chef : You didnt give me any text or Ingredients")
        continue


    multimodal_question = HumanMessage(content=message_content)
    config = {"configurable": {"thread_id": "1"}}

    response = agent.invoke(
        {"messages" : [multimodal_question]},
        config
    )

    img_b64 = None
    aud_b64 = None

    print(response['messages'][-1].content)



[{'type': 'text', 'text': 'Here is a recipe for Chicken Biryani without Biryani Masala, following your available ingredients and rules:\n\n**Ingredients:**\n\n**For Chicken Gravy:**\n*   Oil: 1 Cup\n*   Onions: 2 Cups, thinly sliced\n*   Ginger Paste: 2 Tablespoons\n*   Garlic Paste: 2 Tablespoons\n*   Bay Leaves: 3 pieces, halved\n*   Cinnamon Sticks: 4-5 pieces (approx. 1-inch each)\n*   Green Cardamom: 5-6 pods\n*   Black Cardamom: 2 pods\n*   Red Chilli Powder: 2 Teaspoons\n*   Salt: 2 Teaspoons (adjust to taste)\n*   Turmeric Powder: 1 Teaspoon\n*   Cumin Powder: 1 Teaspoon\n*   Coriander Powder: 1 Teaspoon\n*   Sugar: 1 Teaspoon\n*   Lemon Juice: 2 Tablespoons\n*   Garam Masala Powder: 1 Teaspoon\n*   Poppy Seeds & Mace Powder: 2 Teaspoons (Mace can be crushed from whole blades)\n*   Nutmeg Powder: 1/4 Teaspoon\n*   Water: 1/2 Cup\n*   Tomatoes: 4 Medium, cut into quarters\n*   Chicken: 10 pieces (drumsticks or mixed pieces, about 1-1.5 kg)\n\n**For Soaking Chicken (Optional, for